1. Pearson correlation of asset returns

In [1]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Example daily returns of 5 assets
# ---------------------------------------------------------

returns = pd.DataFrame({
    "Stock_A": [0.010, 0.015, -0.005, 0.008, 0.012,
                -0.010, 0.006, 0.009, -0.004, 0.011],

    "Stock_B": [0.009, 0.014, -0.004, 0.007, 0.011,
                -0.009, 0.005, 0.008, -0.003, 0.010],

    "Stock_C": [-0.005, 0.008, 0.012, -0.010, 0.006,
                0.004, -0.007, 0.011, -0.002, 0.009],

    "Stock_D": [-0.008, -0.012, 0.005, -0.006, -0.010,
                0.009, -0.004, -0.007, 0.003, -0.009],

    "Stock_E": [0.002, -0.004, 0.006, 0.001, -0.003,
                0.005, -0.002, 0.004, 0.000, -0.001]
})


# ---------------------------------------------------------
# Calculate Pearson correlation matrix
# ---------------------------------------------------------

correlation_matrix = returns.corr(method="pearson")

print("Pearson Correlation Matrix:")
print(correlation_matrix)

Pearson Correlation Matrix:
          Stock_A   Stock_B   Stock_C   Stock_D   Stock_E
Stock_A  1.000000  0.999084 -0.015181 -0.998413 -0.678464
Stock_B  0.999084  1.000000  0.005913 -0.998845 -0.685790
Stock_C -0.015181  0.005913  1.000000  0.010606  0.204184
Stock_D -0.998413 -0.998845  0.010606  1.000000  0.691383
Stock_E -0.678464 -0.685790  0.204184  0.691383  1.000000


2. Make a diversification decision

In [2]:
def diversification_decision(correlation):

    if correlation > 0.70:
        return "LIMITED DIVERSIFICATION"

    elif correlation > 0.30:
        return "MODERATE DIVERSIFICATION"

    elif correlation >= -0.30:
        return "GOOD DIVERSIFICATION"

    else:
        return "STRONG DIVERSIFICATION"


# ---------------------------------------------------------
# Evaluate every pair of assets
# ---------------------------------------------------------

assets = correlation_matrix.columns

results = []

for i in range(len(assets)):

    for j in range(i + 1, len(assets)):

        asset_1 = assets[i]
        asset_2 = assets[j]

        corr = correlation_matrix.loc[asset_1, asset_2]

        decision = diversification_decision(corr)

        results.append({
            "Asset 1": asset_1,
            "Asset 2": asset_2,
            "Correlation": round(corr, 3),
            "Decision": decision
        })


diversification_df = pd.DataFrame(results)

print("\nDiversification Analysis:")
print(diversification_df)


Diversification Analysis:
   Asset 1  Asset 2  Correlation                  Decision
0  Stock_A  Stock_B        0.999   LIMITED DIVERSIFICATION
1  Stock_A  Stock_C       -0.015      GOOD DIVERSIFICATION
2  Stock_A  Stock_D       -0.998    STRONG DIVERSIFICATION
3  Stock_A  Stock_E       -0.678    STRONG DIVERSIFICATION
4  Stock_B  Stock_C        0.006      GOOD DIVERSIFICATION
5  Stock_B  Stock_D       -0.999    STRONG DIVERSIFICATION
6  Stock_B  Stock_E       -0.686    STRONG DIVERSIFICATION
7  Stock_C  Stock_D        0.011      GOOD DIVERSIFICATION
8  Stock_C  Stock_E        0.204      GOOD DIVERSIFICATION
9  Stock_D  Stock_E        0.691  MODERATE DIVERSIFICATION


3. Automatically select diversified assets

Suppose you don't want assets with correlation greater than 0.70.

In [3]:
def select_diversified_assets(
    returns,
    correlation_threshold=0.70
):

    correlation_matrix = returns.corr()

    selected_assets = []

    for asset in correlation_matrix.columns:

        # First asset
        if len(selected_assets) == 0:
            selected_assets.append(asset)
            continue

        # Correlation of current asset
        # with already selected assets
        correlations = correlation_matrix.loc[
            asset,
            selected_assets
        ]

        # Check maximum absolute correlation
        max_correlation = correlations.abs().max()

        if max_correlation <= correlation_threshold:

            selected_assets.append(asset)

    return selected_assets


selected = select_diversified_assets(
    returns,
    correlation_threshold=0.70
)

print("\nSelected Assets:")
print(selected)


Selected Assets:
['Stock_A', 'Stock_C', 'Stock_E']


4. Complete portfolio decision function

You can combine everything into one reusable function:

In [4]:
import pandas as pd


def portfolio_diversification_analysis(
    returns,
    correlation_threshold=0.70
):

    # -----------------------------------------
    # Step 1: Pearson correlation
    # -----------------------------------------

    corr_matrix = returns.corr(method="pearson")

    # -----------------------------------------
    # Step 2: Pairwise analysis
    # -----------------------------------------

    pair_results = []

    assets = corr_matrix.columns

    for i in range(len(assets)):

        for j in range(i + 1, len(assets)):

            asset_1 = assets[i]
            asset_2 = assets[j]

            corr = corr_matrix.loc[
                asset_1,
                asset_2
            ]

            # Decision
            if corr > 0.70:

                decision = "REJECT / LIMITED DIVERSIFICATION"

            elif corr > 0.30:

                decision = "MODERATE DIVERSIFICATION"

            elif corr >= -0.30:

                decision = "GOOD DIVERSIFICATION"

            else:

                decision = "STRONG DIVERSIFICATION"

            pair_results.append({
                "Asset_1": asset_1,
                "Asset_2": asset_2,
                "Correlation": round(corr, 3),
                "Decision": decision
            })

    pair_df = pd.DataFrame(pair_results)

    # -----------------------------------------
    # Step 3: Select diversified assets
    # -----------------------------------------

    selected_assets = []

    for asset in assets:

        if not selected_assets:

            selected_assets.append(asset)
            continue

        correlations = corr_matrix.loc[
            asset,
            selected_assets
        ]

        max_abs_corr = correlations.abs().max()

        if max_abs_corr <= correlation_threshold:

            selected_assets.append(asset)

    return corr_matrix, pair_df, selected_assets

In [5]:
corr_matrix, pair_analysis, selected_assets = \
    portfolio_diversification_analysis(
        returns,
        correlation_threshold=0.70
    )


print("========== CORRELATION MATRIX ==========")
print(corr_matrix)

print("\n========== PAIR ANALYSIS ==========")
print(pair_analysis)

print("\n========== SELECTED ASSETS ==========")
print(selected_assets)

========== CORRELATION MATRIX ==========
          Stock_A   Stock_B   Stock_C   Stock_D   Stock_E
Stock_A  1.000000  0.999084 -0.015181 -0.998413 -0.678464
Stock_B  0.999084  1.000000  0.005913 -0.998845 -0.685790
Stock_C -0.015181  0.005913  1.000000  0.010606  0.204184
Stock_D -0.998413 -0.998845  0.010606  1.000000  0.691383
Stock_E -0.678464 -0.685790  0.204184  0.691383  1.000000

========== PAIR ANALYSIS ==========
   Asset_1  Asset_2  Correlation                          Decision
0  Stock_A  Stock_B        0.999  REJECT / LIMITED DIVERSIFICATION
1  Stock_A  Stock_C       -0.015              GOOD DIVERSIFICATION
2  Stock_A  Stock_D       -0.998            STRONG DIVERSIFICATION
3  Stock_A  Stock_E       -0.678            STRONG DIVERSIFICATION
4  Stock_B  Stock_C        0.006              GOOD DIVERSIFICATION
5  Stock_B  Stock_D       -0.999            STRONG DIVERSIFICATION
6  Stock_B  Stock_E       -0.686            STRONG DIVERSIFICATION
7  Stock_C  Stock_D        0.011      